# SMART ETL INITIALIZATION


In [1]:
# ============================================================
# STEP 9 - CELL 1
# SMART ETL INITIALIZATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

print("=" * 70)
print("STEP 9 - SMART ETL")
print("=" * 70)

INPUT_ROOT = Path("/kaggle/input")

OUTPUT_DIR = Path(
    "/kaggle/working/step9_smart_etl"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Input root:", INPUT_ROOT)
print("Output directory:", OUTPUT_DIR)

print("\nAvailable datasets:")

for path in INPUT_ROOT.iterdir():
    print("-", path)

STEP 9 - SMART ETL
Input root: /kaggle/input
Output directory: /kaggle/working/step9_smart_etl

Available datasets:
- /kaggle/input/datasets


In [2]:
# ============================================================
# STEP 9 - CELL 2
# LOCATE INPUT DATA
# ============================================================

def find_dataset_file(filename):

    matches = list(
        INPUT_ROOT.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"Cannot find {filename}"
        )

    print(
        f"{filename}:"
    )

    for match in matches:
        print(" -", match)

    if len(matches) > 1:

        raise RuntimeError(
            f"Found multiple copies of {filename}. "
            "Please select the correct dataset version."
        )

    return matches[0]


PRODUCTS_PATH = find_dataset_file(
    "products.csv"
)

REVIEWS_PATH = find_dataset_file(
    "reviews.csv"
)

print("\n✅ Input files located")

products.csv:
 - /kaggle/input/datasets/trnquangtriu/9datasett/products.csv
reviews.csv:
 - /kaggle/input/datasets/trnquangtriu/9datasett/reviews.csv

✅ Input files located


In [3]:
# ============================================================
# STEP 9 - CELL 3
# LOAD RAW DATA
# ============================================================

products = pd.read_csv(
    PRODUCTS_PATH,
    low_memory=False
)

reviews = pd.read_csv(
    REVIEWS_PATH,
    low_memory=False
)

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(
    "Products:",
    products.shape
)

print(
    "Reviews:",
    reviews.shape
)

print("\nPRODUCT COLUMNS")

print(
    products.columns.tolist()
)

print("\nREVIEW COLUMNS")

print(
    reviews.columns.tolist()
)

print("\nPRODUCT SAMPLE")

display(
    products.head(3)
)

print("\nREVIEW SAMPLE")

display(
    reviews.head(3)
)

DATASET OVERVIEW
Products: (164926, 15)
Reviews: (767179, 11)

PRODUCT COLUMNS
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language']

REVIEW COLUMNS
['review_id', 'user_id', 'product_id', 'rating', 'review_title', 'review_text', 'timestamp', 'verified_purchase', 'helpful_vote', 'language', 'source']

PRODUCT SAMPLE


,product_id,product_name,category,subcategory,brand,price,currency,average_rating,review_count,sold_count,description,image_url,product_url,source,language
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbud...,Apple Products,NaN,Apple,99.00,USD,4.7,585624,NaN,NaN,NaN,NaN,amazon,en
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works ...",Amazon Devices,NaN,Amazon,24.99,USD,4.7,542825,NaN,NaN,NaN,NaN,amazon,en
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4...",Home Audio & Theater,NaN,Amazon Basics,8.55,USD,4.7,507202,NaN,NaN,NaN,NaN,amazon,en



REVIEW SAMPLE


,review_id,user_id,product_id,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,language,source
0,review_0000000,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B07N69T6TM,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,1592678549731,True,0.0,en,amazon
1,review_0000001,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B01G8JO5F2,5.0,Excellent!,I love these. They even come with a carry case...,1523093017534,True,0.0,en,amazon
2,review_0000002,amazon_AG2L7H23R5LLKDKLBEF2Q3L2MVDA,amazon_B07CJYMRWM,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,1676601581238,True,0.0,en,amazon


In [4]:
# ============================================================
# STEP 9 - CELL 4
# DATA QUALITY AUDIT
# ============================================================

def audit_dataframe(
    df,
    dataset_name
):

    print("\n" + "=" * 75)

    print(
        "DATA QUALITY AUDIT:",
        dataset_name
    )

    print("=" * 75)

    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Columns:",
        len(df.columns)
    )

    print(
        "Duplicate rows:",
        f"{df.duplicated().sum():,}"
    )

    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    missing = pd.DataFrame({

        "column":
            df.columns,

        "dtype":
            df.dtypes.astype(str).values,

        "missing_count":
            df.isna().sum().values,

        "missing_percent":
            (
                df.isna().mean() * 100
            ).round(2).values,

        "unique_count":
            df.nunique(
                dropna=True
            ).values

    })

    missing = missing.sort_values(

        "missing_percent",

        ascending=False

    )

    print("\nMISSING VALUE REPORT")

    display(
        missing
    )

    # --------------------------------------------------------
    # Marketplace distribution
    # --------------------------------------------------------

    if "source" in df.columns:

        print(
            "\nMARKETPLACE DISTRIBUTION"
        )

        display(

            df["source"]
            .value_counts(
                dropna=False
            )
            .rename_axis("source")
            .reset_index(
                name="count"
            )

        )

    # --------------------------------------------------------
    # Duplicate IDs
    # --------------------------------------------------------

    id_column = {

        "products":
            "product_id",

        "reviews":
            "review_id"

    }.get(
        dataset_name
    )

    if id_column in df.columns:

        print(
            f"\nDUPLICATE {id_column}:",
            f"{df[id_column].duplicated().sum():,}"
        )

    return missing


product_audit = audit_dataframe(

    products,

    "products"

)

review_audit = audit_dataframe(

    reviews,

    "reviews"

)

print("\n✅ Basic data quality audit completed")


DATA QUALITY AUDIT: products
Rows: 164,926
Columns: 15
Duplicate rows: 0

MISSING VALUE REPORT


,column,dtype,missing_count,missing_percent,unique_count
12,product_url,float64,164926,100.00,0
11,image_url,object,163492,99.13,1431
3,subcategory,object,163492,99.13,22
9,sold_count,float64,123351,74.79,600
10,description,object,123351,74.79,30819
2,category,object,1966,1.19,195
4,brand,object,1498,0.91,25751
5,price,float64,1434,0.87,16144
6,currency,object,0,0.00,2
0,product_id,object,0,0.00,164926



MARKETPLACE DISTRIBUTION


,source,count
0,amazon,121917
1,tiki,41575
2,shopee,1434



DUPLICATE product_id: 0

DATA QUALITY AUDIT: reviews
Rows: 767,179
Columns: 11
Duplicate rows: 0

MISSING VALUE REPORT


,column,dtype,missing_count,missing_percent,unique_count
4,review_title,object,31238,4.07,433709
7,verified_purchase,object,31086,4.05,2
8,helpful_vote,float64,31086,4.05,506
5,review_text,object,403,0.05,702765
2,product_id,object,0,0.00,123351
0,review_id,object,0,0.00,767179
1,user_id,object,0,0.00,228315
6,timestamp,int64,0,0.00,764868
3,rating,float64,0,0.00,5
9,language,object,0,0.00,2



MARKETPLACE DISTRIBUTION


,source,count
0,amazon,736093
1,shopee,31086



DUPLICATE review_id: 0

✅ Basic data quality audit completed


In [5]:
# ============================================================
# STEP 9 - CELL 5
# RECOMMENDER-SPECIFIC DATA AUDIT
# ============================================================

report = {}

# ------------------------------------------------------------
# Product checks
# ------------------------------------------------------------

report["total_products"] = len(
    products
)

report["total_reviews"] = len(
    reviews
)

if "product_id" in products.columns:

    report["missing_product_id"] = int(

        products[
            "product_id"
        ].isna().sum()

    )

    report["duplicate_product_id"] = int(

        products[
            "product_id"
        ].duplicated().sum()

    )

if "product_name" in products.columns:

    report["missing_product_name"] = int(

        products[
            "product_name"
        ].isna().sum()

    )

if "price" in products.columns:

    price = pd.to_numeric(

        products["price"],

        errors="coerce"

    )

    report["missing_or_invalid_price"] = int(

        price.isna().sum()

    )

    report["non_positive_price"] = int(

        (price <= 0).sum()

    )

if "average_rating" in products.columns:

    rating = pd.to_numeric(

        products[
            "average_rating"
        ],

        errors="coerce"

    )

    report["invalid_rating"] = int(

        (
            rating.notna()
            &
            (
                (rating < 0)
                |
                (rating > 5)
            )
        ).sum()

    )

# ------------------------------------------------------------
# Review checks
# ------------------------------------------------------------

if "review_id" in reviews.columns:

    report["duplicate_review_id"] = int(

        reviews[
            "review_id"
        ].duplicated().sum()

    )

if "product_id" in reviews.columns:

    known_products = set(

        products[
            "product_id"
        ].dropna()

    )

    report["reviews_without_product"] = int(

        (
            ~reviews[
                "product_id"
            ].isin(
                known_products
            )
        ).sum()

    )

if "rating" in reviews.columns:

    review_rating = pd.to_numeric(

        reviews["rating"],

        errors="coerce"

    )

    report["invalid_review_rating"] = int(

        (
            review_rating.notna()
            &
            (
                (review_rating < 0)
                |
                (review_rating > 5)
            )
        ).sum()

    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

REPORT_PATH = (

    OUTPUT_DIR

    / "smart_etl_initial_audit.json"

)

with open(

    REPORT_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        report,

        f,

        ensure_ascii=False,

        indent=2

    )

print("=" * 70)
print("SMART ETL INITIAL AUDIT")
print("=" * 70)

print(

    json.dumps(

        report,

        ensure_ascii=False,

        indent=2

    )

)

print("\nSaved:", REPORT_PATH)

SMART ETL INITIAL AUDIT
{
  "total_products": 164926,
  "total_reviews": 767179,
  "missing_product_id": 0,
  "duplicate_product_id": 0,
  "missing_product_name": 1,
  "missing_or_invalid_price": 1434,
  "non_positive_price": 0,
  "invalid_rating": 0,
  "duplicate_review_id": 0,
  "reviews_without_product": 0,
  "invalid_review_rating": 0
}

Saved: /kaggle/working/step9_smart_etl/smart_etl_initial_audit.json


# MARKETPLACE-SPECIFIC DATA QUALITY


In [6]:
# ============================================================
# STEP 9 - CELL 5A
# MARKETPLACE-SPECIFIC DATA QUALITY
# ============================================================

import pandas as pd

print("=" * 75)
print("PRODUCT DATA QUALITY BY MARKETPLACE")
print("=" * 75)

product_quality_by_source = (
    products
    .groupby("source", dropna=False)
    .agg(
        total_products=("product_id", "size"),

        missing_name=(
            "product_name",
            lambda x: x.isna().sum()
        ),

        missing_price=(
            "price",
            lambda x: x.isna().sum()
        ),

        missing_brand=(
            "brand",
            lambda x: x.isna().sum()
        ),

        missing_category=(
            "category",
            lambda x: x.isna().sum()
        ),

        missing_description=(
            "description",
            lambda x: x.isna().sum()
        ),

        missing_product_url=(
            "product_url",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

display(product_quality_by_source)


print("\n" + "=" * 75)
print("PRICE AND CURRENCY BY MARKETPLACE")
print("=" * 75)

display(
    products
    .groupby(
        ["source", "currency"],
        dropna=False
    )
    .agg(
        products=("product_id", "size"),

        products_with_price=(
            "price",
            lambda x: x.notna().sum()
        ),

        min_price=("price", "min"),

        median_price=("price", "median"),

        max_price=("price", "max")
    )
    .reset_index()
)


print("\n" + "=" * 75)
print("REVIEW DATA QUALITY BY MARKETPLACE")
print("=" * 75)

review_quality_by_source = (
    reviews
    .groupby("source", dropna=False)
    .agg(
        total_reviews=("review_id", "size"),

        missing_review_text=(
            "review_text",
            lambda x: x.isna().sum()
        ),

        missing_review_title=(
            "review_title",
            lambda x: x.isna().sum()
        ),

        unique_products=(
            "product_id",
            "nunique"
        ),

        unique_users=(
            "user_id",
            "nunique"
        )
    )
    .reset_index()
)

display(review_quality_by_source)

PRODUCT DATA QUALITY BY MARKETPLACE


,source,total_products,missing_name,missing_price,missing_brand,missing_category,missing_description,missing_product_url
0,amazon,121917,1,0,62,1966,121917,121917
1,shopee,1434,0,1434,1434,0,1434,1434
2,tiki,41575,0,0,2,0,0,41575



PRICE AND CURRENCY BY MARKETPLACE


,source,currency,products,products_with_price,min_price,median_price,max_price
0,amazon,USD,121917,121917,0.01,23.0,24996.99
1,shopee,VND,1434,0,NaN,NaN,NaN
2,tiki,VND,41575,41575,1000.00,85000.0,19800000.00



REVIEW DATA QUALITY BY MARKETPLACE


,source,total_reviews,missing_review_text,missing_review_title,unique_products,unique_users
0,amazon,736093,403,152,121917,198427
1,shopee,31086,0,31086,1434,29888


# Kiểm tra schema đầu vào

In [7]:
# ============================================================
# STEP 9 - CELL 6
# VALIDATE INPUT SCHEMA
# ============================================================

PRODUCT_REQUIRED_COLUMNS = [
    "product_id",
    "product_name",
    "source",
    "price",
    "currency",
    "average_rating",
    "review_count"
]

REVIEW_REQUIRED_COLUMNS = [
    "review_id",
    "product_id",
    "rating",
    "source"
]


def check_schema(df, required_columns, name):

    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df.columns))

    if missing:
        print("Missing columns:", missing)
        raise ValueError(
            f"{name}: missing required columns"
        )

    print("✅ Schema valid")


check_schema(
    products,
    PRODUCT_REQUIRED_COLUMNS,
    "PRODUCTS"
)

check_schema(
    reviews,
    REVIEW_REQUIRED_COLUMNS,
    "REVIEWS"
)


PRODUCTS
Rows: 164,926
Columns: 15
✅ Schema valid

REVIEWS
Rows: 767,179
Columns: 11
✅ Schema valid


# Tạo bản sao dữ liệu để xử lý

In [8]:
# ============================================================
# STEP 9 - CELL 7
# CREATE SMART ETL WORKING DATA
# ============================================================

smart_products = products.copy()

smart_reviews = reviews.copy()

print("=" * 65)
print("SMART ETL WORKING DATA")
print("=" * 65)

print(
    "Original products:",
    products.shape
)

print(
    "Smart products:",
    smart_products.shape
)

print(
    "Original reviews:",
    reviews.shape
)

print(
    "Smart reviews:",
    smart_reviews.shape
)

print("\n✅ Original data preserved")

SMART ETL WORKING DATA
Original products: (164926, 15)
Smart products: (164926, 15)
Original reviews: (767179, 11)
Smart reviews: (767179, 11)

✅ Original data preserved


# Chuẩn hóa dữ liệu sản phẩm

In [9]:
# ============================================================
# STEP 9 - CELL 8
# PRODUCT STANDARDIZATION
# ============================================================

import re
import unicodedata
import numpy as np
import pandas as pd


def normalize_text(value):

    if pd.isna(value):
        return ""

    value = unicodedata.normalize(
        "NFC",
        str(value)
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ------------------------------------------------------------
# Product name
# ------------------------------------------------------------

smart_products["smart_product_name"] = (
    smart_products["product_name"]
    .apply(normalize_text)
)


# ------------------------------------------------------------
# Marketplace
# ------------------------------------------------------------

smart_products["smart_source"] = (
    smart_products["source"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# Currency
# ------------------------------------------------------------

smart_products["smart_currency"] = (
    smart_products["currency"]
    .astype("string")
    .str.strip()
    .str.upper()
)

currency_map = {
    "VNĐ": "VND",
    "Đ": "VND",
    "₫": "VND",
    "$": "USD"
}

smart_products["smart_currency"] = (
    smart_products["smart_currency"]
    .replace(currency_map)
)


# ------------------------------------------------------------
# Price
# ------------------------------------------------------------

smart_products["smart_price"] = pd.to_numeric(
    smart_products["price"],
    errors="coerce"
)

smart_products.loc[
    smart_products["smart_price"] <= 0,
    "smart_price"
] = np.nan


# ------------------------------------------------------------
# Rating
# ------------------------------------------------------------

smart_products["smart_rating"] = pd.to_numeric(
    smart_products["average_rating"],
    errors="coerce"
)

smart_products.loc[
    ~smart_products["smart_rating"].between(0, 5),
    "smart_rating"
] = np.nan


# ------------------------------------------------------------
# Review count
# ------------------------------------------------------------

smart_products["smart_review_count"] = pd.to_numeric(
    smart_products["review_count"],
    errors="coerce"
)

smart_products.loc[
    smart_products["smart_review_count"] < 0,
    "smart_review_count"
] = np.nan


print("=" * 65)
print("PRODUCT STANDARDIZATION COMPLETED")
print("=" * 65)

display(
    smart_products[
        [
            "product_id",
            "smart_product_name",
            "smart_source",
            "smart_price",
            "smart_currency",
            "smart_rating",
            "smart_review_count"
        ]
    ].head(10)
)

PRODUCT STANDARDIZATION COMPLETED


,product_id,smart_product_name,smart_source,smart_price,smart_currency,smart_rating,smart_review_count
0,amazon_B07PXGQC1Q,Apple AirPods (2nd Generation) Wireless Earbud...,amazon,99.00,USD,4.7,585624.0
1,amazon_B07KTYJ769,"Amazon Smart Plug, for home automation, Works ...",amazon,24.99,USD,4.7,542825.0
2,amazon_B0BGNG1294,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4...",amazon,8.55,USD,4.7,507202.0
3,amazon_B08CLNX58K,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memo...,amazon,13.40,USD,4.7,402800.0
4,amazon_B08WJSHSLC,Fire TV Stick (3rd Gen) with Alexa Voice Remot...,amazon,44.98,USD,4.7,379565.0
5,amazon_B07HZLHPKP,"Echo Show 5 (1st Gen, 2019 release) -- Smart d...",amazon,79.99,USD,4.6,356248.0
6,amazon_B08XPWDSWW,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,amazon,21.99,USD,4.3,349254.0
7,amazon_B09V1FT19S,SanDisk 256GB Extreme microSDXC UHS-I Memory C...,amazon,28.99,USD,4.8,337225.0
8,amazon_B08F9ZCTCL,"Fire TV Stick Lite, free and live TV, Alexa Vo...",amazon,19.99,USD,4.7,262857.0
9,amazon_B0BSFT117F,SanDisk Cruzer Blade 16GB USB 2.0 Flash Drive,amazon,5.38,USD,4.7,260446.0


# Tạo quy tắc phân loại sản phẩm

In [10]:
# ============================================================
# STEP 9 - CELL 9
# PRODUCT SEMANTIC TAXONOMY
# ============================================================

import re
import json
import numpy as np
import pandas as pd

PRODUCT_TAXONOMY = {

    "headphones": {
        "main": [
            "headphones",
            "headphone",
            "earbuds",
            "earbud",
            "earphones",
            "earphone",
            "headset",
            "tai nghe"
        ],
        "accessory": [
            "ear tips",
            "ear cushions",
            "ear pads",
            "earpad",
            "replacement pads",
            "headphone stand",
            "headphone case",
            "earbud case",
            "charging case",
            "hộp đựng tai nghe",
            "ốp tai nghe",
            "đệm tai nghe"
        ]
    },

    "camera": {
        "main": [
            "digital camera",
            "dslr camera",
            "mirrorless camera",
            "action camera",
            "máy ảnh",
            "máy quay"
        ],
        "accessory": [
            "camera lens",
            "zoom lens",
            "telephoto lens",
            "lens for",
            "camera bag",
            "camera strap",
            "lens cap",
            "camera battery",
            "camera charger",
            "ống kính",
            "túi máy ảnh",
            "dây đeo máy ảnh"
        ]
    },

    "shoes": {
        "main": [
            "running shoes",
            "sports shoes",
            "sneakers",
            "sneaker",
            "giày thể thao",
            "giày chạy bộ",
            "giày sneaker"
        ],
        "accessory": [
            "shoelaces",
            "shoe laces",
            "shoe insoles",
            "shoe cleaner",
            "dây giày",
            "lót giày",
            "miếng lót giày"
        ]
    },

    "mouse": {
        "main": [
            "computer mouse",
            "wireless mouse",
            "wired mouse",
            "gaming mouse",
            "bluetooth mouse",
            "chuột máy tính",
            "chuột không dây",
            "chuột gaming"
        ],
        "accessory": [
            "mouse pad",
            "mousepad",
            "mouse feet",
            "mouse skates",
            "mouse grip tape",
            "lót chuột",
            "bàn di chuột"
        ]
    },

    "laptop": {
        "main": [
            "laptop",
            "notebook computer",
            "máy tính xách tay"
        ],
        "accessory": [
            "laptop sleeve",
            "laptop case",
            "laptop stand",
            "laptop charger",
            "laptop bag",
            "laptop skin",
            "túi laptop",
            "ốp laptop",
            "giá đỡ laptop",
            "sạc laptop"
        ]
    }

}


def contains_term(text, term):

    pattern = (
        r"(?<!\w)"
        + re.escape(term)
        + r"(?!\w)"
    )

    return bool(
        re.search(pattern, text)
    )


def classify_product_type(product_name):

    text = normalize_text(
        product_name
    ).lower()

    if not text:

        return {
            "product_family": "unknown",
            "product_type": "unknown",
            "classification_method": "unclassified",
            "matched_rule": None
        }

    # Ưu tiên phụ kiện trước sản phẩm chính
    for family, rules in PRODUCT_TAXONOMY.items():

        for term in rules["accessory"]:

            if contains_term(text, term):

                return {
                    "product_family": family,
                    "product_type": "accessory",
                    "classification_method": "keyword_rule",
                    "matched_rule": term
                }

    for family, rules in PRODUCT_TAXONOMY.items():

        for term in rules["main"]:

            if contains_term(text, term):

                return {
                    "product_family": family,
                    "product_type": "main_product",
                    "classification_method": "keyword_rule",
                    "matched_rule": term
                }

    return {
        "product_family": "unknown",
        "product_type": "unknown",
        "classification_method": "unclassified",
        "matched_rule": None
    }


print("✅ Product taxonomy initialized")

✅ Product taxonomy initialized


# Phân loại và bổ sung cờ điều kiện giá

In [11]:
# ============================================================
# STEP 9 - CELL 10
# APPLY SEMANTIC ENRICHMENT - FIX DUPLICATE COLUMNS
# ============================================================

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

print("=" * 75)
print("STEP 9 - SEMANTIC PRODUCT ENRICHMENT")
print("=" * 75)


# ============================================================
# 1. CHECK DUPLICATE COLUMNS
# ============================================================

duplicate_columns = (
    smart_products.columns[
        smart_products.columns.duplicated()
    ]
    .tolist()
)

print("Duplicate columns before fix:")

print(duplicate_columns)


# ============================================================
# 2. REMOVE OLD ENRICHMENT COLUMNS
# ============================================================

# Giữ lại một bản của mỗi cột nếu notebook đã bị concat nhiều lần
smart_products = smart_products.loc[
    :,
    ~smart_products.columns.duplicated()
].copy()


# Xóa kết quả phân loại cũ trước khi tạo lại
enrichment_columns = [

    "product_family",

    "product_type",

    "classification_method",

    "matched_rule",

    "has_valid_price",

    "has_valid_currency",

    "price_filter_eligible"

]


smart_products = smart_products.drop(

    columns=enrichment_columns,

    errors="ignore"

)


print("\nColumns after cleanup:")

print(
    len(smart_products.columns)
)


# ============================================================
# 3. CLASSIFY PRODUCTS
# ============================================================

classification_results = (

    smart_products[
        "smart_product_name"
    ]

    .progress_apply(
        classify_product_type
    )

)


classification_df = pd.DataFrame(

    classification_results.tolist(),

    index=smart_products.index

)


# ============================================================
# 4. MERGE NEW ENRICHMENT COLUMNS
# ============================================================

smart_products = pd.concat(

    [

        smart_products,

        classification_df

    ],

    axis=1

)


# ============================================================
# 5. PRICE ELIGIBILITY
# ============================================================

smart_products["has_valid_price"] = (

    smart_products[
        "smart_price"
    ].notna()

)


smart_products["has_valid_currency"] = (

    smart_products[
        "smart_currency"
    ].isin(
        ["USD", "VND"]
    )

)


smart_products["price_filter_eligible"] = (

    smart_products[
        "has_valid_price"
    ]

    &

    smart_products[
        "has_valid_currency"
    ]

)


# ============================================================
# 6. VALIDATE UNIQUE COLUMNS
# ============================================================

assert smart_products.columns.is_unique, (

    "Duplicate columns still exist!"

)


assert len(smart_products) == len(products), (

    "Product row count changed!"

)


print("\n✅ Duplicate columns fixed")

print("✅ Product count preserved")


# ============================================================
# 7. PRODUCT TYPE DISTRIBUTION
# ============================================================

print("\n" + "=" * 75)

print("PRODUCT TYPE DISTRIBUTION")

print("=" * 75)


display(

    smart_products[
        "product_type"
    ].value_counts(
        dropna=False
    )

)


# ============================================================
# 8. PRODUCT FAMILY DISTRIBUTION
# ============================================================

print("\nPRODUCT FAMILY DISTRIBUTION")


display(

    smart_products[
        "product_family"
    ].value_counts(
        dropna=False
    )

)


# ============================================================
# 9. PRICE ELIGIBILITY BY MARKETPLACE
# ============================================================

print("\nPRICE ELIGIBILITY BY MARKETPLACE")


display(

    smart_products

    .groupby(
        "smart_source"
    )

    .agg(

        total_products=(
            "product_id",
            "size"
        ),

        products_with_valid_price=(
            "price_filter_eligible",
            "sum"
        )

    )

    .reset_index()

)


# ============================================================
# 10. CLASSIFICATION SAMPLE
# ============================================================

print("\nCLASSIFICATION SAMPLE")


display(

    smart_products[

        [

            "product_name",

            "product_family",

            "product_type",

            "classification_method",

            "matched_rule"

        ]

    ]

    .sample(

        min(
            20,
            len(smart_products)
        ),

        random_state=42

    )

)


print("\n✅ CELL 10 COMPLETED SUCCESSFULLY")

STEP 9 - SEMANTIC PRODUCT ENRICHMENT
Duplicate columns before fix:
[]

Columns after cleanup:
21


  0%|          | 0/164926 [00:00<?, ?it/s]


✅ Duplicate columns fixed
✅ Product count preserved

PRODUCT TYPE DISTRIBUTION


product_type
unknown         136617
main_product     22501
accessory         5808
Name: count, dtype: int64


PRODUCT FAMILY DISTRIBUTION


...


PRICE ELIGIBILITY BY MARKETPLACE


,smart_source,total_products,products_with_valid_price
0,amazon,121917,121917
1,shopee,1434,0
2,tiki,41575,41575



CLASSIFICATION SAMPLE


,product_name,product_family,product_type,classification_method,matched_rule
47747,"mCover iPearl Hard Shell Case for 2017 11.6"" D...",laptop,main_product,keyword_rule,laptop
124900,Dép tổ ong Vàng thái loại xịn cho nam,unknown,unknown,unclassified,None
126307,Túi xách da bò công sở cao cấp phong cách châu...,unknown,unknown,unclassified,None
95258,Mission Darkness NeoLok Non-Window Faraday Bag...,unknown,unknown,unclassified,None
14211,"LP-E10 Battery Charger Pack, LP 2-Pack Battery...",unknown,unknown,unclassified,None
10003,"Messenger Bag for Men,VASCHY Vintage Water Res...",laptop,main_product,keyword_rule,laptop
6059,9inch Truck GPS Big Touchscreen Trucking GPS X...,unknown,unknown,unclassified,None
127853,Hình Xăm Dán Đầu Lâu QSA025,unknown,unknown,unclassified,None
135264,TÚI TOTE ĐEO CHÉO CHÚ VỊT OM0059,unknown,unknown,unclassified,None
111570,Intel Socket 775 Copper Core/Aluminum Heat Sin...,unknown,unknown,unclassified,None



✅ CELL 10 COMPLETED SUCCESSFULLY


# Data Quality Flags

In [12]:
# ============================================================
# STEP 9 - CELL 11
# DATA QUALITY FLAGS
# ============================================================

def is_missing(value):

    if pd.isna(value):
        return True

    if isinstance(value, str):
        return not value.strip()

    return False


def build_quality_flags(row):

    flags = []

    if is_missing(
        row.get("product_id")
    ):
        flags.append(
            "missing_product_id"
        )

    if not row.get(
        "smart_product_name"
    ):
        flags.append(
            "missing_product_name"
        )

    if pd.isna(
        row.get("smart_price")
    ):
        flags.append(
            "missing_price"
        )

    if is_missing(
        row.get("smart_currency")
    ):
        flags.append(
            "missing_currency"
        )

    if pd.isna(
        row.get("smart_rating")
    ):
        flags.append(
            "missing_rating"
        )

    if is_missing(
        row.get("brand")
    ):
        flags.append(
            "missing_brand"
        )

    if is_missing(
        row.get("category")
    ):
        flags.append(
            "missing_category"
        )

    if is_missing(
        row.get("description")
    ):
        flags.append(
            "missing_description"
        )

    if is_missing(
        row.get("product_url")
    ):
        flags.append(
            "missing_product_url"
        )

    return flags


smart_products["quality_flags"] = (
    smart_products.apply(
        build_quality_flags,
        axis=1
    )
)

smart_products["quality_issue_count"] = (
    smart_products["quality_flags"]
    .apply(len)
)

smart_products["quality_flags_json"] = (
    smart_products["quality_flags"]
    .apply(
        lambda x: json.dumps(
            x,
            ensure_ascii=False
        )
    )
)


print("=" * 75)
print("QUALITY ISSUE DISTRIBUTION")
print("=" * 75)

display(
    smart_products[
        "quality_issue_count"
    ]
    .value_counts()
    .sort_index()
)

QUALITY ISSUE DISTRIBUTION


quality_issue_count
1     41573
2    119893
3      2023
4      1437
Name: count, dtype: int64

# VALIDATE SMART PRODUCTS


In [13]:
# ============================================================
# STEP 9 - CELL 12
# VALIDATE SMART PRODUCTS
# ============================================================

assert len(products) == len(smart_products)

assert len(reviews) == len(smart_reviews)

assert products["product_id"].reset_index(
    drop=True
).equals(

    smart_products["product_id"].reset_index(
        drop=True
    )

)

assert products["price"].reset_index(
    drop=True
).equals(

    smart_products["price"].reset_index(
        drop=True
    )

)

assert products["source"].reset_index(
    drop=True
).equals(

    smart_products["source"].reset_index(
        drop=True
    )

)


required_columns = [

    "smart_product_name",

    "smart_source",

    "smart_currency",

    "smart_price",

    "smart_rating",

    "smart_review_count",

    "product_family",

    "product_type",

    "classification_method",

    "matched_rule",

    "has_valid_price",

    "price_filter_eligible",

    "quality_flags_json",

    "quality_issue_count"

]


missing_columns = [

    col

    for col in required_columns

    if col not in smart_products.columns

]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)


print("✅ Original rows preserved")
print("✅ Original product IDs preserved")
print("✅ Original prices preserved")
print("✅ Original marketplaces preserved")
print("✅ Smart metadata validated")
print("✅ STEP 9A VALIDATION PASSED")

✅ Original rows preserved
✅ Original product IDs preserved
✅ Original prices preserved
✅ Original marketplaces preserved
✅ Smart metadata validated
✅ STEP 9A VALIDATION PASSED


# SMART ETL REPORT


In [14]:
# ============================================================
# STEP 9 - CELL 13
# SMART ETL REPORT
# ============================================================

from collections import Counter

flag_counter = Counter()

for flags in smart_products[
    "quality_flags"
]:

    flag_counter.update(flags)


report = {

    "step": "9A",

    "total_products":
        len(smart_products),

    "total_reviews":
        len(smart_reviews),

    "marketplace_distribution": (
        smart_products["smart_source"]
        .value_counts()
        .to_dict()
    ),

    "product_type_distribution": (
        smart_products["product_type"]
        .value_counts()
        .to_dict()
    ),

    "product_family_distribution": (
        smart_products["product_family"]
        .value_counts()
        .to_dict()
    ),

    "price_filter_eligible_by_source": (
        smart_products
        .groupby("smart_source")[
            "price_filter_eligible"
        ]
        .sum()
        .astype(int)
        .to_dict()
    ),

    "quality_flag_distribution":
        dict(flag_counter),

    "classification_method":
        "keyword_rule",

    "notes": [

        "No missing prices were imputed.",

        "No product URLs were generated.",

        "Original product IDs and prices were preserved.",

        "Taxonomy covers five product families only.",

        "No embeddings or FAISS indexes were rebuilt."

    ]

}


report_path = (
    OUTPUT_DIR
    / "smart_etl_report.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    )
)

{
  "step": "9A",
  "total_products": 164926,
  "total_reviews": 767179,
  "marketplace_distribution": {
    "amazon": 121917,
    "tiki": 41575,
    "shopee": 1434
  },
  "product_type_distribution": {
    "unknown": 136617,
    "main_product": 22501,
    "accessory": 5808
  },
  "product_family_distribution": {
    "unknown": 136617,
    "laptop": 12612,
    "headphones": 9735,
    "camera": 3700,
    "shoes": 1558,
    "mouse": 704
  },
  "price_filter_eligible_by_source": {
    "amazon": 121917,
    "shopee": 0,
    "tiki": 41575
  },
  "quality_flag_distribution": {
    "missing_description": 123351,
    "missing_product_url": 164926,
    "missing_category": 1966,
    "missing_brand": 1498,
    "missing_product_name": 1,
    "missing_price": 1434
  },
  "classification_method": "keyword_rule",
  "notes": [
    "No missing prices were imputed.",
    "No product URLs were generated.",
    "Original product IDs and prices were preserved.",
    "Taxonomy covers five product families o

In [15]:
# ============================================================
# STEP 9 - CELL 15
# AUDIT POTENTIAL PRODUCT CLASSIFICATION ERRORS
# ============================================================

import pandas as pd

print("=" * 75)
print("PRODUCT CLASSIFICATION ERROR AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Laptop main products containing accessory indicators
# ------------------------------------------------------------

laptop_accessory_pattern = (
    r"(?i)"
    r"case|cover|sleeve|bag|stand|charger|"
    r"skin|shell|ốp|túi|giá đỡ|sạc"
)

suspected_laptop_errors = smart_products[
    (smart_products["product_family"] == "laptop")
    &
    (smart_products["product_type"] == "main_product")
    &
    (
        smart_products["smart_product_name"]
        .str.contains(
            laptop_accessory_pattern,
            regex=True,
            na=False
        )
    )
]

print("\nSUSPECTED LAPTOP ERRORS:")
print(len(suspected_laptop_errors))

display(
    suspected_laptop_errors[
        [
            "product_name",
            "product_family",
            "product_type",
            "matched_rule"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 2. Camera accessories containing main camera indicators
# ------------------------------------------------------------

camera_main_pattern = (
    r"(?i)"
    r"dslr camera|mirrorless camera|"
    r"digital camera|action camera|"
    r"máy ảnh|máy quay"
)

suspected_camera_errors = smart_products[
    (smart_products["product_family"] == "camera")
    &
    (smart_products["product_type"] == "accessory")
    &
    (
        smart_products["smart_product_name"]
        .str.contains(
            camera_main_pattern,
            regex=True,
            na=False
        )
    )
]

print("\nSUSPECTED CAMERA ERRORS:")
print(len(suspected_camera_errors))

display(
    suspected_camera_errors[
        [
            "product_name",
            "product_family",
            "product_type",
            "matched_rule"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

audit_summary = pd.DataFrame(
    [
        {
            "check": "Laptop main with accessory indicators",
            "suspected_count": len(suspected_laptop_errors)
        },
        {
            "check": "Camera accessory with main camera indicators",
            "suspected_count": len(suspected_camera_errors)
        }
    ]
)

display(audit_summary)

print("\n✅ Classification error audit completed")

PRODUCT CLASSIFICATION ERROR AUDIT

SUSPECTED LAPTOP ERRORS:
3487


,product_name,product_family,product_type,matched_rule
108,Amazon Basics USB Type-C to USB-A 2.0 Male Cha...,laptop,main_product,laptop
231,External DVD Drive USB 3.0 TYPE C Portable CD/...,laptop,main_product,laptop
280,"CloudValley Webcam Cover Slide[2-Pack], 0.023 ...",laptop,main_product,laptop
304,Tzowla Travel Laptop Backpack Water Resistant ...,laptop,main_product,laptop
349,"YOREPEK Travel Backpack, Extra Large 50L Lapto...",laptop,main_product,laptop
552,LOVEVOOK Laptop Backpack for Women Fashion Tra...,laptop,main_product,laptop
604,"17 Inch Laptop Backpack, Business Anti Theft S...",laptop,main_product,laptop
605,SHRRADOO Anti Theft Laptop Backpack Travel Bac...,laptop,main_product,laptop
667,KROSER Laptop Backpack 15.6 Inch Stylish Backp...,laptop,main_product,laptop
808,"himawari Laptop Backpack for Women&Men,Wide Op...",laptop,main_product,laptop



SUSPECTED CAMERA ERRORS:
423


,product_name,product_family,product_type,matched_rule
3156,Blue Jeans Camera Strap Real Denim for All SLR...,camera,accessory,camera strap
3280,Lens Cap Bundle - 4 Snap-on Lens Caps for DSLR...,camera,accessory,lens cap
3953,Nikon AF FX NIKKOR 50mm f/1.8D Lens for Nikon ...,camera,accessory,lens for
3960,waka 77mm MC UV Filter - Ultra Slim 16 Layers ...,camera,accessory,camera lens
4604,"waka Unique Design Lens Cap Bundle, 3 Pcs 58mm...",camera,accessory,lens cap
4780,Nikon AF-S DX NIKKOR 55-300mm f/4.5-5.6G ED Vi...,camera,accessory,zoom lens
5446,Altura Photo (4 Pack) Thick Protective Neopren...,camera,accessory,camera lens
5474,Sigma 17-50mm f/2.8 EX DC OS HSM FLD Large Ape...,camera,accessory,zoom lens
5740,"62mm Snap-On Center-Pinch Lens Cap, Extra Stro...",camera,accessory,camera lens
6080,"Camera Backpack, BAGSMAR DSLR Camera Bag Backp...",camera,accessory,camera bag


,check,suspected_count
0,Laptop main with accessory indicators,3487
1,Camera accessory with main camera indicators,423



✅ Classification error audit completed


In [16]:
# ============================================================
# STEP 9 - CELL 16
# CREATE MANUAL LABEL VALIDATION SET
# ============================================================

from pathlib import Path

VALIDATION_DIR = (
    OUTPUT_DIR / "validation"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sample_parts = []

for product_type in [
    "main_product",
    "accessory",
    "unknown"
]:

    group = smart_products[
        smart_products["product_type"] == product_type
    ]

    sample = group.sample(
        n=min(100, len(group)),
        random_state=42
    )

    sample_parts.append(sample)


validation_df = pd.concat(
    sample_parts,
    ignore_index=True
)

validation_df = validation_df[
    [
        "product_id",
        "source",
        "product_name",
        "category",
        "product_family",
        "product_type",
        "classification_method",
        "matched_rule"
    ]
].copy()


# Để trống cho người kiểm tra điền
validation_df["true_product_family"] = ""

validation_df["true_product_type"] = ""

validation_df["validation_note"] = ""


validation_path = (
    VALIDATION_DIR
    / "product_type_validation_300.csv"
)

validation_df.to_csv(
    validation_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", validation_path)

print("Total samples:", len(validation_df))

display(
    validation_df.head(10)
)

Saved: /kaggle/working/step9_smart_etl/validation/product_type_validation_300.csv
Total samples: 300


,product_id,source,product_name,category,product_family,product_type,classification_method,matched_rule,true_product_family,true_product_type,validation_note
0,amazon_B09Q6K38G1,amazon,PlayZoom 2 Kids Smartwatch & Headphones - Vide...,All Electronics,headphones,main_product,keyword_rule,headphones,,,
1,amazon_B08V1NQ8S9,amazon,"ASUS VivoBook 17.3"" FHD IPS LED Backlight Prem...",Computers,laptop,main_product,keyword_rule,laptop,,,
2,amazon_B081W2KLB8,amazon,Lens Adapter EF/EF-S to Nikon Z-Mount Mirrorle...,All Electronics,camera,main_product,keyword_rule,mirrorless camera,,,
3,amazon_B017L0P880,amazon,SW The Empire Doesn't Care About Your Stick Fi...,Automotive,laptop,main_product,keyword_rule,laptop,,,
4,amazon_B00BWRO5D8,amazon,Hello Kitty Earbuds - White/Pink (11409-HK),Home Audio & Theater,headphones,main_product,keyword_rule,earbuds,,,
5,amazon_B07FDYF6T3,amazon,"Call Center Telephone with Headset, MCHEETA Ph...",All Electronics,headphones,main_product,keyword_rule,headset,,,
6,amazon_B08YNVD3C5,amazon,"Headphone Extension Cable with Microphone, Hym...",All Electronics,headphones,main_product,keyword_rule,headphone,,,
7,amazon_B09PGRQGCN,amazon,"VIVO 38 inch Desk Converter, Height Adjustable...",Industrial & Scientific,laptop,main_product,keyword_rule,laptop,,,
8,amazon_B09XTT923Y,amazon,Hagibis Cleaning Pen for Airpods Pro 1 2 Multi...,Cell Phones & Accessories,headphones,main_product,keyword_rule,earbuds,,,
9,amazon_B08LKXDBX3,amazon,Belker 90W Laptop Car Charger for HP Dell Asus...,Computers,laptop,main_product,keyword_rule,laptop,,,


# Lưu nhãn V1 và định nghĩa bộ quy tắc V2

In [17]:
# ============================================================
# STEP 9 - CELL 17
# PREPARE PRODUCT CLASSIFICATION V2
# ============================================================

import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Preserve V1 classification
# ------------------------------------------------------------

v1_columns = {
    "product_family": "product_family_v1",
    "product_type": "product_type_v1",
    "classification_method": "classification_method_v1",
    "matched_rule": "matched_rule_v1"
}

for old_col, new_col in v1_columns.items():

    if new_col not in smart_products.columns:

        smart_products[new_col] = (
            smart_products[old_col]
        )


# ------------------------------------------------------------
# Main product indicators
# ------------------------------------------------------------

MAIN_PRODUCT_RULES = {

    "laptop": [
        "laptop computer",
        "notebook computer",
        "gaming laptop",
        "business laptop",
        "chromebook",
        "macbook pro",
        "macbook air",
        "máy tính xách tay"
    ],

    "camera": [
        "dslr camera",
        "mirrorless camera",
        "digital camera",
        "action camera",
        "point and shoot camera",
        "máy ảnh",
        "máy quay"
    ],

    "headphones": [
        "wireless earbuds",
        "bluetooth earbuds",
        "wireless headphones",
        "bluetooth headphones",
        "gaming headset",
        "tai nghe bluetooth",
        "tai nghe không dây"
    ],

    "shoes": [
        "running shoes",
        "sports shoes",
        "sneakers",
        "giày thể thao",
        "giày chạy bộ"
    ],

    "mouse": [
        "wireless mouse",
        "gaming mouse",
        "bluetooth mouse",
        "computer mouse",
        "chuột không dây",
        "chuột máy tính"
    ]

}


# ------------------------------------------------------------
# Accessory indicators
# ------------------------------------------------------------

ACCESSORY_RULES = {

    "laptop": [
        "laptop backpack",
        "laptop bag",
        "laptop sleeve",
        "laptop case",
        "laptop cover",
        "laptop stand",
        "laptop charger",
        "laptop skin",
        "hard shell case",
        "backpack for laptop",
        "bag for laptop",
        "case for laptop",
        "charger for laptop",
        "túi laptop",
        "balo laptop",
        "ốp laptop",
        "sạc laptop"
    ],

    "camera": [
        "camera lens",
        "zoom lens",
        "telephoto lens",
        "lens cap",
        "camera strap",
        "camera bag",
        "camera backpack",
        "camera battery",
        "camera charger",
        "lens for",
        "ống kính",
        "túi máy ảnh",
        "dây đeo máy ảnh"
    ],

    "headphones": [
        "earbud case",
        "headphone case",
        "charging case",
        "ear tips",
        "ear pads",
        "ear cushions",
        "headphone stand",
        "ốp tai nghe",
        "hộp đựng tai nghe"
    ],

    "shoes": [
        "shoelaces",
        "shoe laces",
        "shoe insoles",
        "shoe cleaner",
        "dây giày",
        "lót giày"
    ],

    "mouse": [
        "mouse pad",
        "mousepad",
        "mouse feet",
        "mouse skates",
        "lót chuột",
        "bàn di chuột"
    ]

}

print("✅ V1 preserved")
print("✅ V2 classification rules initialized")

✅ V1 preserved
✅ V2 classification rules initialized


# Bộ phân loại V2 có xét ngữ cảnh

In [18]:
# ============================================================
# STEP 9 - CELL 18
# CONTEXT-AWARE PRODUCT CLASSIFICATION V2
# ============================================================

def find_matched_terms(text, terms):

    matches = []

    for term in terms:

        pattern = (
            r"(?<!\w)"
            + re.escape(term)
            + r"(?!\w)"
        )

        if re.search(pattern, text):

            matches.append(term)

    return matches


def classify_product_v2(product_name):

    text = normalize_text(
        product_name
    ).lower()

    if not text:

        return {
            "product_family_v2": "unknown",
            "product_type_v2": "unknown",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": "missing_product_name"
        }

    family_results = []

    for family in MAIN_PRODUCT_RULES:

        main_matches = find_matched_terms(
            text,
            MAIN_PRODUCT_RULES[family]
        )

        accessory_matches = find_matched_terms(
            text,
            ACCESSORY_RULES[family]
        )

        if main_matches or accessory_matches:

            family_results.append({

                "family": family,

                "main_matches": main_matches,

                "accessory_matches": accessory_matches

            })

    if not family_results:

        return {
            "product_family_v2": "unknown",
            "product_type_v2": "unknown",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": "no_matching_rule"
        }

    # Multiple product families detected
    if len(family_results) > 1:

        return {
            "product_family_v2": "unknown",
            "product_type_v2": "needs_review",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": "multiple_product_families"
        }

    result = family_results[0]

    family = result["family"]

    main_matches = result["main_matches"]

    accessory_matches = result["accessory_matches"]

    # --------------------------------------------------------
    # Accessory only
    # --------------------------------------------------------

    if accessory_matches and not main_matches:

        return {
            "product_family_v2": family,
            "product_type_v2": "accessory",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": (
                "accessory_only: "
                + ", ".join(accessory_matches)
            )
        }

    # --------------------------------------------------------
    # Main product only
    # --------------------------------------------------------

    if main_matches and not accessory_matches:

        return {
            "product_family_v2": family,
            "product_type_v2": "main_product",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": (
                "main_only: "
                + ", ".join(main_matches)
            )
        }

    # --------------------------------------------------------
    # Both main and accessory indicators
    # --------------------------------------------------------

    if main_matches and accessory_matches:

        # Clear accessory constructions:
        # "case for DSLR camera", "bag for laptop", etc.

        accessory_for_main = re.search(
            r"\b(case|bag|cover|sleeve|"
            r"charger|strap|lens cap)\b"
            r".{0,45}\bfor\b",
            text
        )

        if accessory_for_main:

            return {
                "product_family_v2": family,
                "product_type_v2": "accessory",
                "classification_method_v2": "rule_based_v2",
                "classification_reason_v2": "accessory_for_main_product"
            }

        # Main camera bundle:
        # "DSLR Camera with 18-55mm Zoom Lens"

        if family == "camera":

            camera_bundle = re.search(
                r"\b(dslr|mirrorless|digital|action)"
                r"\s+camera\b"
                r".{0,65}\bwith\b",
                text
            )

            if camera_bundle:

                return {
                    "product_family_v2": "camera",
                    "product_type_v2": "main_product",
                    "classification_method_v2": "rule_based_v2",
                    "classification_reason_v2": "camera_bundle"
                }

        return {
            "product_family_v2": family,
            "product_type_v2": "needs_review",
            "classification_method_v2": "rule_based_v2",
            "classification_reason_v2": "conflicting_main_accessory_signals"
        }

    return {
        "product_family_v2": "unknown",
        "product_type_v2": "unknown",
        "classification_method_v2": "rule_based_v2",
        "classification_reason_v2": "unclassified"
    }


print("✅ Context-aware classifier V2 ready")

✅ Context-aware classifier V2 ready


# Chạy phân loại V2

In [19]:
# ============================================================
# STEP 9 - CELL 19
# APPLY PRODUCT CLASSIFICATION V2
# ============================================================

from tqdm.auto import tqdm

tqdm.pandas()

v2_columns = [
    "product_family_v2",
    "product_type_v2",
    "classification_method_v2",
    "classification_reason_v2"
]

# Cho phép chạy lại cell mà không tạo cột trùng
smart_products = smart_products.drop(
    columns=v2_columns,
    errors="ignore"
)

v2_results = (
    smart_products["smart_product_name"]
    .progress_apply(classify_product_v2)
)

v2_df = pd.DataFrame(
    v2_results.tolist(),
    index=smart_products.index
)

smart_products = pd.concat(
    [smart_products, v2_df],
    axis=1
)

assert smart_products.columns.is_unique

assert len(smart_products) == len(products)


print("=" * 75)
print("V1 VS V2 PRODUCT TYPE")
print("=" * 75)

comparison = pd.DataFrame({

    "V1": smart_products[
        "product_type_v1"
    ].value_counts(),

    "V2": smart_products[
        "product_type_v2"
    ].value_counts()

}).fillna(0).astype(int)

display(comparison)


print("\nV2 PRODUCT FAMILY")

display(
    smart_products[
        "product_family_v2"
    ].value_counts()
)


print("\nV1 → V2 TRANSITION")

display(
    pd.crosstab(
        smart_products["product_type_v1"],
        smart_products["product_type_v2"]
    )
)


print("\nSAMPLE CHANGED LABELS")

changed = smart_products[
    smart_products["product_type_v1"]
    !=
    smart_products["product_type_v2"]
]

display(
    changed[
        [
            "product_name",
            "product_type_v1",
            "product_type_v2",
            "product_family_v2",
            "classification_reason_v2"
        ]
    ].sample(
        min(25, len(changed)),
        random_state=42
    )
)

print("\nTotal changed labels:", len(changed))

  0%|          | 0/164926 [00:00<?, ?it/s]

V1 VS V2 PRODUCT TYPE


,V1,V2
accessory,5808,5925
main_product,22501,8640
needs_review,0,1190
unknown,136617,149171



V2 PRODUCT FAMILY


product_family_v2
unknown       149332
laptop          7257
camera          3737
headphones      2876
shoes           1185
mouse            539
Name: count, dtype: int64


V1 → V2 TRANSITION


product_type_v2,accessory,main_product,needs_review,unknown
product_type_v1,,,,
accessory,4763,105,892,48
main_product,1009,6252,161,15079
unknown,153,2283,137,134044



SAMPLE CHANGED LABELS


,product_name,product_type_v1,product_type_v2,product_family_v2,classification_reason_v2
53302,BASN in-Ear Monitor Headphones Dual Dynamic Dr...,main_product,unknown,unknown,no_matching_rule
88142,TLZC 30L Flight Approved Weekender Carry-On La...,main_product,unknown,unknown,no_matching_rule
20531,Cuero Handmade Vintage Unisex Leather Laptop M...,main_product,unknown,unknown,no_matching_rule
38406,Microsoft Surface Pen for Surface Pro 7 Pro 6 ...,main_product,unknown,unknown,no_matching_rule
14207,"[2021 Updated] Extra Long Earbuds for TV & PC,...",main_product,unknown,unknown,no_matching_rule
65663,Jabra Motion UC Bluetooth Headset - Retail Pac...,main_product,unknown,unknown,no_matching_rule
61933,Panasonic RP-HT21 Lightweight Headphones with ...,main_product,unknown,unknown,no_matching_rule
83315,AreMe USB C to Magnetic T-Tip Charging Adapter...,unknown,main_product,laptop,main_only: macbook pro
42744,Plugfones Guardian Plus in-Ear Earplug Earbud ...,main_product,unknown,unknown,no_matching_rule
18102,Bluetooth Headphones with 4 Mics Clear Call St...,accessory,needs_review,headphones,conflicting_main_accessory_signals



Total changed labels: 19867


In [20]:
# ============================================================
# STEP 9 - CELL 20
# REVIEW PREVIOUS SUSPECTED ERRORS
# ============================================================

print("=" * 75)
print("LAPTOP SUSPECTED ERRORS - V2")
print("=" * 75)

display(
    smart_products.loc[
        suspected_laptop_errors.index,
        [
            "product_name",
            "product_type_v1",
            "product_type_v2",
            "classification_reason_v2"
        ]
    ].head(20)
)


print("\n" + "=" * 75)
print("CAMERA SUSPECTED ERRORS - V2")
print("=" * 75)

display(
    smart_products.loc[
        suspected_camera_errors.index,
        [
            "product_name",
            "product_type_v1",
            "product_type_v2",
            "classification_reason_v2"
        ]
    ].head(20)
)


print("\nV2 LABEL DISTRIBUTION IN SUSPECTED GROUPS")

for name, indices in [

    (
        "laptop_suspected",
        suspected_laptop_errors.index
    ),

    (
        "camera_suspected",
        suspected_camera_errors.index
    )

]:

    print("\n", name)

    display(
        smart_products.loc[
            indices,
            "product_type_v2"
        ].value_counts(
            dropna=False
        )
    )

LAPTOP SUSPECTED ERRORS - V2


,product_name,product_type_v1,product_type_v2,classification_reason_v2
108,Amazon Basics USB Type-C to USB-A 2.0 Male Cha...,main_product,unknown,no_matching_rule
231,External DVD Drive USB 3.0 TYPE C Portable CD/...,main_product,unknown,no_matching_rule
280,"CloudValley Webcam Cover Slide[2-Pack], 0.023 ...",main_product,main_product,main_only: macbook pro
304,Tzowla Travel Laptop Backpack Water Resistant ...,main_product,accessory,accessory_only: laptop backpack
349,"YOREPEK Travel Backpack, Extra Large 50L Lapto...",main_product,unknown,no_matching_rule
552,LOVEVOOK Laptop Backpack for Women Fashion Tra...,main_product,accessory,accessory_only: laptop backpack
604,"17 Inch Laptop Backpack, Business Anti Theft S...",main_product,accessory,accessory_only: laptop backpack
605,SHRRADOO Anti Theft Laptop Backpack Travel Bac...,main_product,accessory,accessory_only: laptop backpack
667,KROSER Laptop Backpack 15.6 Inch Stylish Backp...,main_product,accessory,accessory_only: laptop backpack
808,"himawari Laptop Backpack for Women&Men,Wide Op...",main_product,accessory,accessory_only: laptop backpack



CAMERA SUSPECTED ERRORS - V2


,product_name,product_type_v1,product_type_v2,classification_reason_v2
3156,Blue Jeans Camera Strap Real Denim for All SLR...,accessory,accessory,accessory_for_main_product
3280,Lens Cap Bundle - 4 Snap-on Lens Caps for DSLR...,accessory,accessory,accessory_only: lens cap
3953,Nikon AF FX NIKKOR 50mm f/1.8D Lens for Nikon ...,accessory,accessory,accessory_only: lens for
3960,waka 77mm MC UV Filter - Ultra Slim 16 Layers ...,accessory,needs_review,conflicting_main_accessory_signals
4604,"waka Unique Design Lens Cap Bundle, 3 Pcs 58mm...",accessory,accessory,accessory_for_main_product
4780,Nikon AF-S DX NIKKOR 55-300mm f/4.5-5.6G ED Vi...,accessory,accessory,accessory_only: zoom lens
5446,Altura Photo (4 Pack) Thick Protective Neopren...,accessory,needs_review,conflicting_main_accessory_signals
5474,Sigma 17-50mm f/2.8 EX DC OS HSM FLD Large Ape...,accessory,needs_review,conflicting_main_accessory_signals
5740,"62mm Snap-On Center-Pinch Lens Cap, Extra Stro...",accessory,accessory,"accessory_only: camera lens, lens cap"
6080,"Camera Backpack, BAGSMAR DSLR Camera Bag Backp...",accessory,accessory,accessory_for_main_product



V2 LABEL DISTRIBUTION IN SUSPECTED GROUPS

 laptop_suspected


product_type_v2
unknown         2464
main_product     584
accessory        408
needs_review      31
Name: count, dtype: int64


 camera_suspected


product_type_v2
accessory       220
needs_review    104
main_product     99
Name: count, dtype: int64

## STEP 9B — Bổ sung để phục vụ chatbot Top 5

Các cell sau chạy sau V2; không giả định nhãn V2 đã đúng. Không tự sinh giá, URL, review hoặc chuyển đổi USD/VND. Báo cáo và file cuối chỉ xuất ở cell cuối.

In [21]:
# CELL 21 — CATEGORY DIAGNOSTICS (KHÔNG ĐOÁN CATEGORY)
import json, re, shutil
from collections import Counter
import numpy as np
import pandas as pd

category_distribution = (smart_products.groupby(["smart_source", "category"], dropna=False)
    .size().reset_index(name="product_count")
    .sort_values("product_count", ascending=False))
category_by_v2 = (smart_products.groupby(["smart_source", "category", "product_type_v2"], dropna=False)
    .size().reset_index(name="product_count")
    .sort_values("product_count", ascending=False))
category_distribution.to_csv(OUTPUT_DIR / "category_distribution.csv", index=False, encoding="utf-8-sig")
category_by_v2.to_csv(OUTPUT_DIR / "category_classification_v2.csv", index=False, encoding="utf-8-sig")
print("TOP 30 CATEGORY:")
display(category_distribution.head(30))
print("V2 LABEL DISTRIBUTION:")
display(smart_products["product_type_v2"].value_counts(dropna=False))
print("Lưu ý: 'unknown' có thể do taxonomy chỉ phủ 5 nhóm; 'needs_review' không phải nhãn đã kiểm chứng.")

TOP 30 CATEGORY:


,smart_source,category,product_count
2,amazon,All Electronics,37102
16,amazon,Computers,28707
155,tiki,Root,16595
13,amazon,Camera & Photo,16108
15,amazon,Cell Phones & Accessories,11166
23,amazon,Home Audio & Theater,7300
24,amazon,Industrial & Scientific,5167
161,tiki,Thời Trang,4230
33,amazon,Tools & Home Improvement,2454
36,amazon,NaN,1966


V2 LABEL DISTRIBUTION:


product_type_v2
unknown         149171
main_product      8640
accessory         5925
needs_review      1190
Name: count, dtype: int64

Lưu ý: 'unknown' có thể do taxonomy chỉ phủ 5 nhóm; 'needs_review' không phải nhãn đã kiểm chứng.


# REVIEW ETL: CHỈ CHUẨN HÓA, KHÔNG TẠO REVIEW MỚI

In [22]:
# CELL 22 — REVIEW ETL: CHỈ CHUẨN HÓA, KHÔNG TẠO REVIEW MỚI
import unicodedata

def safe_text(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(v))).strip()

smart_reviews = reviews.copy()
smart_reviews["smart_review_text"] = (smart_reviews["review_text"].map(safe_text)
    if "review_text" in smart_reviews.columns else "")
smart_reviews["smart_review_rating"] = pd.to_numeric(smart_reviews["rating"], errors="coerce")
smart_reviews.loc[~smart_reviews["smart_review_rating"].between(1, 5), "smart_review_rating"] = np.nan
smart_reviews["smart_source"] = smart_reviews["source"].astype("string").str.strip().str.lower()
smart_reviews["has_review_evidence"] = smart_reviews["smart_review_text"].str.len().gt(0)
smart_reviews["review_evidence_eligible"] = (smart_reviews["has_review_evidence"]
    & smart_reviews["smart_review_rating"].notna())
assert len(smart_reviews) == len(reviews)
assert smart_reviews.columns.is_unique
print("REVIEW COVERAGE BY MARKETPLACE")
display(smart_reviews.groupby("smart_source", dropna=False).agg(
    total_reviews=("review_id", "size"),
    text_available=("has_review_evidence", "sum"),
    usable_text_and_rating=("review_evidence_eligible", "sum")))
print("Không đồng nhất số review trong file với review_count trên sàn.")

REVIEW COVERAGE BY MARKETPLACE


,total_reviews,text_available,usable_text_and_rating
smart_source,,,
amazon,736093,735690,735690
shopee,31086,31086,31086


Không đồng nhất số review trong file với review_count trên sàn.


# READINESS FOR TOP-5 RAG: JOIN THEO product_id, KHÔNG JOIN THEO VỊ TRÍ VECTOR

In [23]:
# CELL 23 — READINESS FOR TOP-5 RAG: JOIN THEO product_id, KHÔNG JOIN THEO VỊ TRÍ VECTOR
assert smart_products["product_id"].is_unique, "Trùng product_id: cần xử lý trước khi join"
assert smart_reviews["review_id"].is_unique, "Trùng review_id: cần kiểm tra"
assert smart_reviews["product_id"].isin(smart_products["product_id"]).all(), "Có review không tìm thấy sản phẩm"
review_coverage = smart_reviews.groupby("product_id").agg(
    etl_review_rows=("review_id", "size"),
    etl_review_text_rows=("has_review_evidence", "sum"),
    etl_review_evidence_rows=("review_evidence_eligible", "sum")
).reset_index()
smart_products = smart_products.drop(columns=[
    "etl_review_rows", "etl_review_text_rows", "etl_review_evidence_rows",
    "has_review_evidence", "top5_evidence_ready", "product_url_available",
    "product_type_candidate", "product_family_candidate", "classification_requires_validation"
], errors="ignore")
smart_products = smart_products.merge(review_coverage, how="left", on="product_id", validate="one_to_one")
for col in ["etl_review_rows", "etl_review_text_rows", "etl_review_evidence_rows"]:
    smart_products[col] = smart_products[col].fillna(0).astype("int64")
smart_products["has_review_evidence"] = smart_products["etl_review_evidence_rows"].gt(0)
smart_products["product_url_available"] = (smart_products["product_url"].map(safe_text)
    .str.match(r"^https?://[^\s]+$", case=False, na=False))
smart_products["product_type_candidate"] = smart_products["product_type_v2"]
smart_products["product_family_candidate"] = smart_products["product_family_v2"]
smart_products["classification_requires_validation"] = smart_products["product_type_v2"].isin(["unknown", "needs_review"])
# Chỉ thể hiện độ sẵn sàng metadata/evidence, KHÔNG khẳng định review nói đúng về thuộc tính được hỏi
smart_products["top5_evidence_ready"] = (smart_products["smart_product_name"].str.len().gt(0)
    & smart_products["price_filter_eligible"] & smart_products["has_review_evidence"])
assert len(smart_products) == len(products)
assert smart_products.columns.is_unique
print("PRODUCT READINESS BY SOURCE")
display(smart_products.groupby("smart_source").agg(
    products=("product_id", "size"),
    priced=("price_filter_eligible", "sum"),
    with_review_text=("has_review_evidence", "sum"),
    with_url=("product_url_available", "sum"),
    top5_evidence_ready=("top5_evidence_ready", "sum")))
print("Cảnh báo: USD và VND không được lọc chung một ngưỡng ngân sách.")

PRODUCT READINESS BY SOURCE


,products,priced,with_review_text,with_url,top5_evidence_ready
smart_source,,,,,
amazon,121917,121917,121876,0,121875
shopee,1434,0,1434,0,0
tiki,41575,41575,0,0,0


Cảnh báo: USD và VND không được lọc chung một ngưỡng ngân sách.


# MANUAL VALIDATION (CHỈ TÍNH METRIC KHI ĐÃ ĐIỀN NHÃN THẬT)

In [24]:
# CELL 24 — MANUAL VALIDATION (CHỈ TÍNH METRIC KHI ĐÃ ĐIỀN NHÃN THẬT)
validation_path = OUTPUT_DIR / "validation" / "product_type_validation_300.csv"
if validation_path.exists():
    labels = pd.read_csv(validation_path, dtype="string").fillna("")
    if "true_product_type" in labels.columns:
        labeled = labels[labels["true_product_type"].str.strip().ne("")].copy()
        print(f"Manually labeled: {len(labeled)} / {len(labels)}")
        if len(labeled):
            compare = labeled[["product_id", "true_product_type"]].merge(
                smart_products[["product_id", "product_type_v1", "product_type_v2"]],
                on="product_id", how="inner", validate="one_to_one")
            for version in ["v1", "v2"]:
                pred = compare[f"product_type_{version}"]
                print(version, "accuracy on labeled sample:", round(float(
                    pred.eq(compare["true_product_type"]).mean()), 4))
            compare.to_csv(OUTPUT_DIR / "validation" / "manual_validation_comparison.csv",
                           index=False, encoding="utf-8-sig")
        else:
            print("Chưa có nhãn thật: KHÔNG báo cáo accuracy/precision/recall.")
else:
    print("Chưa có validation CSV. Chạy cell tạo mẫu thủ công trước.")
print("Lưu ý: mẫu 100/nhóm V1 là stratified, không đại diện tự nhiên cho toàn bộ dataset.")

Manually labeled: 0 / 300
Chưa có nhãn thật: KHÔNG báo cáo accuracy/precision/recall.
Lưu ý: mẫu 100/nhóm V1 là stratified, không đại diện tự nhiên cho toàn bộ dataset.


# FINAL REPORT + EXPORT (CHẠY CUỐI CÙNG)

In [25]:
# CELL 25 — FINAL REPORT + EXPORT (CHẠY CUỐI CÙNG)
from collections import Counter
flag_counter = Counter()
for flags in smart_products["quality_flags"]: flag_counter.update(flags)
report_final = {
    "step": "9", "status": "smart_etl_v1_v2_candidate_not_ground_truth",
    "input": {"products": int(len(products)), "reviews": int(len(reviews))},
    "output": {"smart_products": int(len(smart_products)), "smart_reviews": int(len(smart_reviews))},
    "products_by_source": {k:int(v) for k,v in smart_products["smart_source"].value_counts().items()},
    "reviews_by_source": {k:int(v) for k,v in smart_reviews["smart_source"].value_counts().items()},
    "product_type_v1": {k:int(v) for k,v in smart_products["product_type_v1"].value_counts().items()},
    "product_type_v2": {k:int(v) for k,v in smart_products["product_type_v2"].value_counts().items()},
    "quality_flags": dict(flag_counter),
    "readiness_by_source": smart_products.groupby("smart_source")[
        ["price_filter_eligible", "has_review_evidence", "product_url_available", "top5_evidence_ready"]
    ].sum().astype(int).to_dict(orient="index"),
    "limitations": ["Only five families in V1/V2 taxonomy", "V2 labels not manually validated",
                    "No prices/URLs/reviews imputed", "No FX conversion", "No embeddings/FAISS rebuilt",
                    "Knowledge Editing not implemented in Step 9"]
}
with open(OUTPUT_DIR / "smart_etl_report.json", "w", encoding="utf-8") as f:
    json.dump(report_final, f, ensure_ascii=False, indent=2)
export_products = smart_products.drop(columns=["quality_flags"], errors="ignore")
assert export_products.columns.is_unique and smart_reviews.columns.is_unique
assert export_products["product_id"].is_unique
for name, frame in [("smart_products", export_products), ("smart_reviews", smart_reviews)]:
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    frame.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
zip_path = shutil.make_archive("/kaggle/working/step9_smart_etl", "zip",
                               root_dir="/kaggle/working", base_dir="step9_smart_etl")
print("FINAL REPORT:")
print(json.dumps(report_final, ensure_ascii=False, indent=2))
print("ZIP:", zip_path)
print("STEP 9 FINAL EXPORT COMPLETED")

FINAL REPORT:
{
  "step": "9",
  "status": "smart_etl_v1_v2_candidate_not_ground_truth",
  "input": {
    "products": 164926,
    "reviews": 767179
  },
  "output": {
    "smart_products": 164926,
    "smart_reviews": 767179
  },
  "products_by_source": {
    "amazon": 121917,
    "tiki": 41575,
    "shopee": 1434
  },
  "reviews_by_source": {
    "amazon": 736093,
    "shopee": 31086
  },
  "product_type_v1": {
    "unknown": 136617,
    "main_product": 22501,
    "accessory": 5808
  },
  "product_type_v2": {
    "unknown": 149171,
    "main_product": 8640,
    "accessory": 5925,
    "needs_review": 1190
  },
  "quality_flags": {
    "missing_description": 123351,
    "missing_product_url": 164926,
    "missing_category": 1966,
    "missing_brand": 1498,
    "missing_product_name": 1,
    "missing_price": 1434
  },
  "readiness_by_source": {
    "amazon": {
      "price_filter_eligible": 121917,
      "has_review_evidence": 121876,
      "product_url_available": 0,
      "top5_evidenc